# Phase 04: Prescriptive Data Analysis

Objective: Investigating our results and infering business decisions.

In [70]:
import pandas as pd

In [71]:
frame = pd.read_csv('../data/intermediate/probabilities.csv', index_col='customer_id')
frame.head(5)

,probability,monthly_charge
customer_id,,
9242-TKFSV,0.021632,65.1
4009-ALQFH,0.512231,99.5
8184-WMOFI,0.044770,71.4
3612-YUNGG,0.078485,109.2
4307-KTUMW,0.512759,93.9


## Expected Value

Objective: Calculate the expected value of retention to prioritize interventions.

Inputs: ML-generated churn probabilities and existing monthly charges.

Assumptions: A 6-month expected retention period and a €10 intervention cost (e.g., a promotional discount).

Outcome: Identifying which specific customers to target while estimating the total intervention budget.

In [72]:
retention_months = 6        # how long we expect to retain customers 
cost_of_intervention = 10   # cost of discount or new offer in €

In [73]:
# calculating possible revenue by customer
frame['revenue'] = frame['probability'] * frame['monthly_charge'] * retention_months
frame['expected_value'] = frame['revenue'] - cost_of_intervention
frame['intervene'] = frame['expected_value'] > 0

intervene            = frame[frame['intervene'] == True].shape[0]
no_intervene         = frame[frame['intervene'] == False].shape[0]
total_intervene_cost = frame.loc[(frame['intervene'] == True), 'expected_value'].sum()

In [74]:
print(f"Number of customers we should:")
print(f"\nIntervene:     {intervene}")
print(f"\nNot intervene:  {no_intervene}")
print(f"\nTotal cost of intervention:")

total_intervene_cost_in_k = round((total_intervene_cost / 1000), 2)

print(f"\n€{total_intervene_cost_in_k}k")

Number of customers we should:

Intervene:     1112

Not intervene:  273

Total cost of intervention:

€186.69k


In [75]:
pseudo = pd.read_csv('../data/intermediate/post-eda.csv')
pseudo = pseudo[['customer_id', 'customer_status']]
pseudo = frame.merge(pseudo, how='left', on='customer_id')
pseudo = pseudo[pseudo['customer_status'] != 'Churned']
pseudo = pseudo.drop(columns=('customer_status'))

# customers with highest expected values
pseudo.sort_values(by='expected_value', ascending=False).round(2).head(5)

,customer_id,probability,monthly_charge,revenue,expected_value,intervene
295,7379-FNIUJ,0.90,100.20,542.24,532.24,True
233,5647-FXOTP,0.80,105.90,510.47,500.47,True
702,9094-AZPHK,0.85,100.15,508.27,498.27,True
969,7577-SWIFR,0.93,89.25,498.21,488.21,True
1383,9129-UXERG,0.80,103.60,497.53,487.53,True


While this approach successfully identifies high-priority customers for retention, further financial modeling is constrained by current data limitations.

Incorporating granular data (such as customer-specific intervention costs) would allow us to expand this analysis into a comprehensive budget optimization model.